# Q-HPC — Step 8 CUDA Execution

Runs the Performance Engine's **CUDA paths** on a real GPU and records
predicted-vs-actual telemetry.

Your laptop has no NVIDIA GPU, so these two code paths cannot be exercised
there at all:

- `XGBoost` with `device="cuda"`
- `qiskit-aer` statevector simulation with `device="GPU"`

Everything else in Step 8 (CPU, MPI) runs better on your laptop — 16 cores
against Colab's 2. This notebook is only for the parts that need the GPU.

**First:** Runtime → Change runtime type → T4 GPU → Save.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi

## 2. Install dependencies

`qiskit-aer-gpu` replaces `qiskit-aer` and carries the CUDA statevector
backend. Having both installed causes an import conflict, so the CPU wheel
goes first.

In [ ]:
!pip install -q pydantic psutil numpy scipy
!pip install -q xgboost
!pip uninstall -q -y qiskit-aer
!pip install -q qiskit qiskit-aer-gpu
print("install step finished")

In [ ]:
# Report the real error for anything that will not import.
for name in ("numpy", "scipy", "xgboost", "qiskit", "qiskit_aer",
             "pydantic", "psutil"):
    try:
        mod = __import__(name)
        print(f"  OK    {name:<12} {getattr(mod, '__version__', '?')}")
    except Exception as exc:
        print(f"  FAIL  {name:<12} {type(exc).__name__}: {exc}")

In [ ]:
# Confirm qiskit-aer can actually see the GPU. If this prints an empty
# list, the GPU build did not install and every GPU simulator run below
# will fail — which is the correct, visible outcome rather than a silent
# fallback to CPU.
from qiskit_aer import AerSimulator
print("available devices:", AerSimulator().available_devices())

## 3. Get the Q-HPC code

Use Option A if `backend/models/` is in the repo by now. Otherwise zip your
local `backend/` folder and use Option B.

In [ ]:
# OPTION A — clone the repo
!git clone -b dev https://github.com/ronsmessi1012/QHPC.git /content/QHPC
%cd /content/QHPC
!ls backend backend/engine backend/models

In [ ]:
# OPTION B — upload a zip of your local backend/ folder.
# Use this if Option A is missing backend/models/ or backend/engine/.
#
# from google.colab import files
# import zipfile, os
# os.makedirs("/content/QHPC", exist_ok=True)
# up = files.upload()                       # select backend.zip
# for fn in up:
#     with zipfile.ZipFile(fn) as z:
#         z.extractall("/content/QHPC")
# %cd /content/QHPC
# !ls backend backend/engine backend/models

## 4. Hardware detection should now find the GPU

Same `hardware_detect.py` that reports "no CUDA device" on your laptop.

In [ ]:
!python -m backend.utils.hardware_detect

## 5. Run the Step 8 test suite

The optional-dependency tests assert correct behaviour either way: with
xgboost and qiskit present here, the `succeeds_or_explains_itself` tests
should now take their **success** branch, where they check the correctness
metrics rather than the error messages.

In [ ]:
!pip install -q pytest
!python -m pytest backend/tests/test_performance_engine.py -v

## 6. XGBoost — CPU vs CUDA through the real pipeline

This runs Steps 6 → 7 → 8 twice over the same problem, forcing the backend
each time, and compares. The estimator, mapper and engine are the project's
own code; nothing here is a special-case benchmark script.

In [ ]:
from backend.models.problem_schema import ProblemSpecification
from backend.models.decision_schema import ExecutionDecision
from backend.models.hardware_schema import HardwareMapping
from backend.agents.resource_estimator import estimate_resources
from backend.engine.performance_engine import execute
from backend.utils.hardware_detect import detect_hardware

inventory = detect_hardware()
print(f"CUDA: {inventory.gpu_description}\n")

rows = []
for dataset_size in (10_000, 100_000, 500_000):
    spec = ProblemSpecification(
        user_query=f"classify {dataset_size} records",
        problem_class="classification", confidence=0.85,
        dimensionality=50, dataset_size=dataset_size,
        constraint_structure="unconstrained")
    decision = ExecutionDecision(
        selected_family="classical", selected_algorithm="XGBoost",
        routing_mode="classical_only", decision_confidence=0.85,
        borderline_score=0.20, explanation="benchmark")

    estimate = estimate_resources(decision, spec, inventory=inventory)

    result = {}
    for backend in ("cpu", "gpu_cuda"):
        mapping = HardwareMapping(
            backend_type=backend, device_spec=f"forced {backend}",
            justification="explicit comparison", reasoning_trace={})
        t = execute(mapping, estimate, decision, spec)
        result[backend] = t
        status = "ok" if t.success else f"FAILED: {t.error_message[:60]}"
        print(f"  n={dataset_size:>7,}  {backend:<9} "
              f"{t.actual_runtime_seconds:>8.3f} s  "
              f"acc={t.result_metric_value if t.success else float('nan'):.4f}  {status}")

    if result["cpu"].success and result["gpu_cuda"].success:
        speedup = (result["cpu"].actual_runtime_seconds
                   / result["gpu_cuda"].actual_runtime_seconds)
        rows.append((dataset_size, speedup,
                     estimate.estimated_runtime_seconds,
                     result["cpu"].actual_runtime_seconds,
                     result["gpu_cuda"].actual_runtime_seconds))
        print(f"  {'':>9} speedup  {speedup:>8.2f}x\n")

print()
print(f"{'n':>9}  {'CPU (s)':>9}  {'GPU (s)':>9}  {'speedup':>8}  {'predicted (s)':>13}")
print("-" * 58)
for n, s, pred, c, g in rows:
    print(f"{n:>9,}  {c:>9.3f}  {g:>9.3f}  {s:>7.2f}x  {pred:>13.3f}")

### What this tells you about `GPU_SPEEDUP_FACTOR`

`resource_constants.py` currently assumes `6.0`. Compare that to the
measured column above.

Two things to notice, both worth stating in the write-up:

1. **Speedup grows with input size.** Kernel launch and host-to-device
   transfer are fixed costs, so small inputs show little or no gain — and
   can be slower on GPU than CPU. A single scalar factor is a
   simplification; say which size it was measured at.

2. **This number does not transfer.** It is XGBoost on a T4 against Colab's
   CPU. On different hardware it would differ, and your laptop cannot
   produce it at all.

## 7. Quantum simulation — CPU vs GPU statevector

Grover search through the real engine, on both devices. The correctness
metric is the probability mass on the marked state, so a run that produced
a wrong answer fast would still be visible as wrong.

In [ ]:
from backend.models.resource_schema import ResourceEstimate

print(f"{'qubits':>7}  {'CPU (s)':>9}  {'GPU (s)':>9}  {'speedup':>8}  "
      f"{'P(marked)':>10}")
print("-" * 52)

for n_qubits in (10, 14, 18, 20, 22):
    dataset_size = 2 ** n_qubits
    spec = ProblemSpecification(
        user_query=f"search {dataset_size} items", problem_class="search",
        confidence=0.9, dimensionality=n_qubits, dataset_size=dataset_size,
        constraint_structure="unconstrained")
    decision = ExecutionDecision(
        selected_family="quantum", selected_algorithm="Grover Search",
        routing_mode="quantum_only", decision_confidence=0.85,
        borderline_score=0.20, explanation="benchmark")

    estimate = estimate_resources(decision, spec, inventory=inventory)
    mapping = HardwareMapping(
        backend_type="qpu_simulator", device_spec="statevector",
        justification="benchmark", reasoning_trace={})

    t_cpu = execute(mapping, estimate, decision, spec, use_gpu_simulator=False)
    t_gpu = execute(mapping, estimate, decision, spec, use_gpu_simulator=True)

    if t_cpu.success and t_gpu.success:
        sp = t_cpu.actual_runtime_seconds / t_gpu.actual_runtime_seconds
        print(f"{n_qubits:>7}  {t_cpu.actual_runtime_seconds:>9.3f}  "
              f"{t_gpu.actual_runtime_seconds:>9.3f}  {sp:>7.2f}x  "
              f"{t_gpu.result_metric_value:>10.4f}")
    else:
        failed = t_cpu if not t_cpu.success else t_gpu
        which = "CPU" if not t_cpu.success else "GPU"
        print(f"{n_qubits:>7}  {which} failed: {failed.error_message[:50]}")

## 8. The thesis table: predicted vs measured

This is the result worth reporting. Steps 6 and 7 claim to predict cost and
choose a backend; this measures whether the prediction held.

`runtime_error_ratio` is actual ÷ predicted. A ratio of 1.0 means the model
was right; 0.1 means it was ten times pessimistic; 10 means ten times
optimistic.

All current constants are uncalibrated, so expect large ratios. That is the
finding, not a failure — and `compared_against_uncalibrated_estimate`
records it so the table cannot be misread as a mis-prediction by a measured
model.

In [ ]:
import json

CASES = [
    ("classical", "XGBoost",                "cpu",           50,  100_000),
    ("classical", "XGBoost",                "gpu_cuda",      50,  100_000),
    ("classical", "Monte Carlo Simulation", "cpu",           10,    5_000),
    ("quantum",   "Grover Search",          "qpu_simulator", 18,  262_144),
    ("quantum",   "Quantum Approximate Optimization Algorithm (QAOA)",
                                            "qpu_simulator", 24,       24),
]

records = []
print(f"{'algorithm':<26} {'backend':<14} {'pred (s)':>9} {'act (s)':>9} "
      f"{'ratio':>8} {'metric':>10}")
print("-" * 82)

for family, algorithm, backend, dim, ds in CASES:
    spec = ProblemSpecification(
        user_query="telemetry run", problem_class="optimization",
        confidence=0.8, dimensionality=dim, dataset_size=ds,
        constraint_structure="constrained")
    decision = ExecutionDecision(
        selected_family=family, selected_algorithm=algorithm,
        routing_mode="borderline_review", decision_confidence=0.70,
        borderline_score=0.90, explanation="telemetry run")

    estimate = estimate_resources(decision, spec, inventory=inventory)
    mapping = HardwareMapping(
        backend_type=backend, device_spec=f"forced {backend}",
        justification="telemetry run",
        reasoning_trace={"mpi_slots": 1})

    t = execute(mapping, estimate, decision, spec,
                use_gpu_simulator=(backend == "qpu_simulator"))
    records.append(t.model_dump())

    if t.success:
        ratio = f"{t.runtime_error_ratio:.3f}" if t.runtime_error_ratio else "n/a"
        print(f"{algorithm[:25]:<26} {backend:<14} "
              f"{t.predicted_runtime_seconds:>9.3f} "
              f"{t.actual_runtime_seconds:>9.3f} {ratio:>8} "
              f"{t.result_metric_value:>10.4f}")
    else:
        print(f"{algorithm[:25]:<26} {backend:<14} FAILED: "
              f"{t.error_message[:34]}")

with open("/content/step8_telemetry.json", "w") as fh:
    json.dump(records, fh, indent=2)
print("\nwrote /content/step8_telemetry.json")

In [ ]:
# Summary of how well the cost model held up.
ok = [r for r in records if r["success"] and r["runtime_error_ratio"]]
if ok:
    import statistics
    ratios = [r["runtime_error_ratio"] for r in ok]
    print(f"runs with a usable ratio : {len(ok)}")
    print(f"median runtime ratio     : {statistics.median(ratios):.3f}")
    print(f"range                    : {min(ratios):.3f} to {max(ratios):.3f}")
    within2 = sum(1 for r in ratios if 0.5 <= r <= 2.0)
    print(f"within 2x of prediction  : {within2}/{len(ok)}")
    uncal = sum(1 for r in ok if r["compared_against_uncalibrated_estimate"])
    print(f"\n{uncal}/{len(ok)} compared against UNCALIBRATED estimates —")
    print("so these ratios measure an unmeasured model, not a bad one.")
else:
    print("no successful runs with a usable ratio")

In [ ]:
from google.colab import files
files.download("/content/step8_telemetry.json")

---

## What you can honestly claim after running this

- **The CUDA execution path works** — XGBoost `device="cuda"` and
  qiskit-aer GPU statevector both run through the project's own engine, with
  correctness metrics proving they computed the right answer.
- **A measured GPU speedup**, at a stated input size, on a stated device.
- **Predicted vs measured telemetry** across backends, with the
  uncalibrated-model caveat recorded per run.

## What you still cannot claim

- Any GPU result for your laptop. There is no NVIDIA GPU in it.
- MPI scaling from here. Colab has ~2 cores; your laptop's 16 is the right
  machine for that, and `--oversubscribe` numbers would be meaningless.
- Multi-node distributed execution. Nothing in this project has been tested
  across more than one machine.

## Next

1. Save `step8_telemetry.json` with the thesis — it is the evidence.
2. Feed the measured values back into `resource_constants.py`, setting
   `calibrated: True` and `calibration_host` to this Colab machine.
3. Do the MPI half on your laptop once Microsoft MPI or WSL2 is sorted.